# Estate Monitor: demo traffic simulator
Streams realistic, **clearly labelled synthetic events** (`data.simulated = true`, source `estate-monitor/simulator`) so the Real-Time Estate Monitor tells a story even in a quiet demo tenant. It uses the same event shapes as the native Fabric events and the collector, and the kit's own workspaces, items and personas.

| Minute | Story beat | Where it shows |
|---|---|---|
| continuous | Directors and the executive view the hub; occasional Copilot use; scheduled notebook, pipeline and refresh jobs | Live operations, Activity and security |
| 3 to 7 | The slow visual's query storm pushes the capacity past 100%; capacity state goes **Overloaded**, then recovers | Capacity tiles, Top capacity consumers |
| 5 | **Monthly Member Intelligence Hub** refresh fails in 2-TEST-QA (missing warehouse) | Live job feed, Failed jobs card |
| 8 to 10 | `mgmt_analyst` exports about 20 times in two minutes | Export and download watch |
| 11 | `contractor_temp` becomes **Admin** of 3-PROD | Workspace access changes |
| 12 | A report copy appears in 1-DEV; the model is redeployed to 3-PROD | Change feed |

Start it about five minutes before Act 7. Change `BEATS` to move or skip beats. To remove simulated data afterwards, see doc 07.


## Parameters

In [ ]:
# ---- Where events go (same options as the collector) ----
SINK = "eventstream"                 # "eventstream", "eventhouse" or "dryrun"
EVENTSTREAM_CONNECTION_STRING = ""
KEY_VAULT_URI = ""
EVENTSTREAM_SECRET_NAME = ""
EVENTHOUSE_QUERY_URI = ""
KQL_DATABASE = "EstateMonitor"
KQL_TABLE = "RawEvents"
AUTH_MODE = "user"; TENANT_ID = ""; CLIENT_ID = ""; CLIENT_SECRET_NAME = ""

# ---- Scenario ----
TENANT_DOMAIN = "contoso.example"    # persona UPNs become west_director@<this>, etc.
DURATION_MINUTES = 20                # how long to stream
TICK_SECONDS = 5
SEND_BASELINE = True                 # inventory and access baseline for the demo workspaces
CAPACITY_NAME = "demo-capacity (simulated)"
CAPACITY_SKU, BASE_CU = "F64", 64

# Minutes after start when each story beat fires. Set any to None to skip it.
BEATS = {
    "capacity_spike_start": 3,       # a query storm from the slow visual pushes the capacity past 100%
    "capacity_spike_end": 7,
    "refresh_failure": 5,            # Monthly Member Intelligence Hub refresh fails in 2-TEST-QA
    "export_burst_start": 8,         # one analyst exports about 20 times in two minutes
    "export_burst_end": 10,
    "new_admin": 11,                 # an unexpected account becomes Admin of 3-PROD
    "content_changes": 12,           # a report copy appears in 1-DEV, the model is redeployed to 3-PROD
}
RANDOM_SEED = 7


## Helpers

In [ ]:
# ---- Shared helpers: identity, REST calls, event envelope, sinks ----
import json, time, uuid, hmac, hashlib, base64, urllib.parse, os, copy, random, math
from datetime import datetime, timezone, timedelta
import requests

try:
    nbu = notebookutils                      # Fabric runtime global
except NameError:                            # older runtimes
    from notebookutils import mssparkutils as nbu

def utcnow():
    return datetime.now(timezone.utc)

def iso(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

def _secret(name):
    if not name:
        return ""
    if not KEY_VAULT_URI:
        raise ValueError(f"Set KEY_VAULT_URI to read the secret '{name}'.")
    return nbu.credentials.getSecret(KEY_VAULT_URI, name)

class Identity:
    """Tokens for the Fabric, Power BI and Kusto APIs, as the notebook user or a service principal."""
    SCOPES = {"pbi": "https://analysis.windows.net/powerbi/api/.default",
              "fabric": "https://api.fabric.microsoft.com/.default"}
    def __init__(self):
        self._cache = {}
        self._app = None
        if AUTH_MODE == "service_principal":
            import msal
            secret = _secret(CLIENT_SECRET_NAME)
            self._app = msal.ConfidentialClientApplication(
                CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}", client_credential=secret)

    def token(self, resource):
        hit = self._cache.get(resource)
        if hit and hit[1] > time.time() + 120:
            return hit[0]
        if self._app is None:
            if resource == "kusto":
                try:
                    tok = nbu.credentials.getToken("kusto")
                except Exception:
                    tok = nbu.credentials.getToken(EVENTHOUSE_QUERY_URI.rstrip("/"))
            else:
                tok = nbu.credentials.getToken("pbi")      # accepted by the Power BI and Fabric APIs
        else:
            scope = self.SCOPES.get(resource) or f"{EVENTHOUSE_QUERY_URI.rstrip('/')}/.default"
            res = self._app.acquire_token_for_client(scopes=[scope])
            if "access_token" not in res:
                raise RuntimeError(f"Token request failed: {res.get('error_description', res)}")
            tok = res["access_token"]
        self._cache[resource] = (tok, time.time() + 600)
        return tok

def make_event(etype, data, subject="", event_id=None, when=None, capacity_id="", source="estate-monitor/collector"):
    return {"specversion": "1.0", "id": event_id or str(uuid.uuid4()), "source": source, "type": etype,
            "subject": subject, "time": when or iso(utcnow()), "dataschemaversion": "1.0",
            "capacityId": capacity_id or "", "data": data}

class EventstreamSink:
    """Sends events to an Eventstream custom endpoint (Event Hub-compatible) over HTTPS with a SAS key.
    No extra packages needed."""
    MAX_BATCH_BYTES = 900_000
    MAX_BATCH_EVENTS = 500
    def __init__(self, connection_string):
        parts = dict(p.split("=", 1) for p in connection_string.strip().strip(";").split(";") if "=" in p)
        self.host = parts["Endpoint"].replace("sb://", "").strip("/")
        self.entity = parts["EntityPath"]
        self.key_name = parts["SharedAccessKeyName"]
        self.key = parts["SharedAccessKey"]
        self.url = f"https://{self.host}/{self.entity}/messages?timeout=60&api-version=2014-01"

    def _sas(self, ttl=3600):
        uri = urllib.parse.quote_plus(f"https://{self.host}/{self.entity}")
        expiry = str(int(time.time() + ttl))
        sig = base64.b64encode(hmac.new(self.key.encode("utf-8"), f"{uri}\n{expiry}".encode("utf-8"), hashlib.sha256).digest())
        return f"SharedAccessSignature sr={uri}&sig={urllib.parse.quote_plus(sig)}&se={expiry}&skn={self.key_name}"

    def _post(self, batch):
        body = json.dumps([{"Body": json.dumps(e, default=str)} for e in batch])
        for attempt in range(5):
            r = requests.post(self.url, data=body.encode("utf-8"), timeout=60,
                              headers={"Authorization": self._sas(),
                                       "Content-Type": "application/vnd.microsoft.servicebus.json"})
            if r.status_code in (200, 201):
                return
            if r.status_code in (429, 500, 503) and attempt < 4:
                time.sleep(2 ** attempt); continue
            raise RuntimeError(f"Eventstream send failed: HTTP {r.status_code} {r.text[:300]}")

    def send(self, events):
        batch, size = [], 2
        for e in events:
            n = len(json.dumps(e, default=str)) + 16
            if batch and (size + n > self.MAX_BATCH_BYTES or len(batch) >= self.MAX_BATCH_EVENTS):
                self._post(batch); batch, size = [], 2
            batch.append(e); size += n
        if batch:
            self._post(batch)
        return len(events)

class EventhouseSink:
    """Writes events straight into the RawEvents table with the Spark Kusto connector (no Eventstream)."""
    COLS = ["id", "source", "type", "subject", "time", "specversion", "dataschemaversion", "capacityId", "data"]
    def __init__(self, identity):
        if not EVENTHOUSE_QUERY_URI:
            raise ValueError("Set EVENTHOUSE_QUERY_URI for SINK = 'eventhouse'.")
        self.identity = identity
    def send(self, events):
        if not events:
            return 0
        from pyspark.sql.types import StructType, StructField, StringType
        rows = [tuple(json.dumps(e["data"], default=str) if c == "data" else str(e.get(c) or "") for c in self.COLS) for e in events]
        df = spark.createDataFrame(rows, StructType([StructField(c, StringType()) for c in self.COLS]))
        (df.write.format("com.microsoft.kusto.spark.synapse.datasource")
           .option("kustoCluster", EVENTHOUSE_QUERY_URI)
           .option("kustoDatabase", KQL_DATABASE)
           .option("kustoTable", KQL_TABLE)
           .option("accessToken", self.identity.token("kusto"))
           .option("adjustSchema", "GenerateDynamicCsvMapping")
           .mode("Append").save())
        return len(events)

class DryRunSink:
    """Prints a sample instead of sending. Use SINK = 'dryrun' to test API access first."""
    def __init__(self): self.sent = []
    def send(self, events):
        self.sent.extend(events)
        for e in events[:3]:
            print("  sample:", json.dumps(e, default=str)[:300])
        return len(events)

def build_sink(identity):
    if SINK == "eventstream":
        cs = EVENTSTREAM_CONNECTION_STRING or _secret(EVENTSTREAM_SECRET_NAME)
        if not cs:
            raise ValueError("Set EVENTSTREAM_CONNECTION_STRING, or KEY_VAULT_URI + EVENTSTREAM_SECRET_NAME.")
        return EventstreamSink(cs)
    if SINK == "eventhouse":
        return EventhouseSink(identity)
    if SINK == "dryrun":
        return DryRunSink()
    raise ValueError("SINK must be 'eventstream', 'eventhouse' or 'dryrun'.")


## Simulate

In [ ]:
random.seed(RANDOM_SEED)
identity = Identity() if SINK == "eventhouse" else None
sink = build_sink(identity)
SRC = "estate-monitor/simulator"
NS = uuid.UUID("0b8e5c1d-3f7a-4c2e-9d61-5a4b3c2d1e0f")
def gid(label): return str(uuid.uuid5(NS, label))

CAP_ID = gid("capacity")
WORKSPACES = ["1-DEV", "2-TEST-QA", "3-PROD"]
ITEMS = [("Contoso_CU", "Lakehouse"), ("Fabrikam_CU", "Lakehouse"), ("Enterprise_Ref", "Lakehouse"),
         ("Combined-Data", "Warehouse"), ("Monthly Member Intelligence Hub", "SemanticModel"),
         ("Monthly Member Intelligence Hub", "Report"), ("Load_Source_Data", "Notebook"),
         ("Nightly combined load", "DataPipeline")]
def ws_id(ws): return gid("ws:" + ws)
def item_id(ws, name, kind): return gid(f"item:{ws}:{name}:{kind}")
def upn(p): return f"{p}@{TENANT_DOMAIN}"
DIRECTORS = ["west_director", "east_director", "north_director", "south_director"]
VIEWERS = DIRECTORS + ["retail_exec"]
REPORT = "Monthly Member Intelligence Hub"

def ev(etype, data, subject="", when=None):
    data = {**data, "simulated": True}
    return make_event(etype, data, subject, when=iso(when or utcnow()), capacity_id=CAP_ID, source=SRC)

def item_data(ws, name, kind):
    return {"itemId": item_id(ws, name, kind), "itemName": name, "itemKind": kind,
            "workspaceId": ws_id(ws), "workspaceName": ws}

def baseline():
    out = []
    for ws in WORKSPACES:
        out.append(ev("Estate.Inventory.WorkspaceDiscovered", {"workspaceId": ws_id(ws), "workspaceName": ws,
                      "workspaceType": "Workspace", "state": "Active", "capacityId": CAP_ID}, f"/workspaces/{ws_id(ws)}"))
        for name, kind in ITEMS:
            d = item_data(ws, name, kind)
            out.append(ev("Estate.Inventory.ItemDiscovered", {**d, "workspaceType": "Workspace", "capacityId": CAP_ID,
                          "state": "Active", "lastUpdatedDate": iso(utcnow() - timedelta(days=random.randint(1, 40))),
                          "createdBy": upn("mgmt_analyst")}, f"/workspaces/{ws_id(ws)}/items/{d['itemId']}"))
        roles = [(upn("fabric_admin"), "Admin"), (upn("mgmt_analyst"), "Contributor" if ws == "1-DEV" else "Viewer")] + \
                [(upn(p), "Viewer") for p in VIEWERS]
        for principal, role in roles:
            out.append(ev("Estate.Access.Discovered", {"workspaceId": ws_id(ws), "workspaceName": ws, "principal": principal,
                          "principalType": "User", "role": role}, f"/workspaces/{ws_id(ws)}"))
    out.append(ev("Estate.Inventory.WorkspaceDiscovered", {"workspaceId": ws_id("Sandbox-Departed"), "workspaceName": "Sandbox-Departed",
                  "workspaceType": "Workspace", "state": "Active", "capacityId": CAP_ID}, "/workspaces/" + ws_id("Sandbox-Departed")))
    return out

def job_events(ws, name, kind, job_type, invoke, start, seconds, status="Completed", error=""):
    """Created -> InProgress -> Succeeded/Failed, as Fabric job events (or refresh events for semantic models)."""
    d = item_data(ws, name, kind)
    jid = str(uuid.uuid4())
    base = {**d, "jobInstanceId": jid, "jobType": job_type, "jobInvokeType": invoke,
            "executingPrincipalType": "User", "jobScheduleTime": iso(start), "jobStartTime": iso(start)}
    subj = f"/workspaces/{d['workspaceId']}/items/{d['itemId']}/jobs/instances/{jid}"
    end = start + timedelta(seconds=seconds)
    if kind == "SemanticModel":                       # semantic model refreshes come from the collector in real life
        return [(start, ev("Estate.Refresh", {**base, "jobStatus": "Unknown"}, subj, start)),
                (end, ev("Estate.Refresh", {**base, "jobStatus": status, "jobEndTime": iso(end), "errorMessage": error}, subj, end))]
    final = "Microsoft.Fabric.ItemJobSucceeded" if status == "Completed" else "Microsoft.Fabric.ItemJobFailed"
    return [(start, ev("Microsoft.Fabric.ItemJobCreated", {**base, "jobStatus": "NotStarted"}, subj, start)),
            (start + timedelta(seconds=2), ev("Microsoft.Fabric.ItemJobStatusChanged", {**base, "jobStatus": "InProgress"}, subj, start)),
            (end, ev(final, {**base, "jobStatus": status, "jobEndTime": iso(end), "failureReason": error}, subj, end))]

def operation(ws, name, kind, op_name, util_type, cu_ms, dur_ms, who, when, throttle_ms=0):
    d = item_data(ws, name, kind)
    return ev("Microsoft.Fabric.CapacityOperationEvents.Operation",
              {**d, "capacityId": CAP_ID, "capacityName": CAPACITY_NAME, "capacityFriendlyName": CAPACITY_NAME,
               "capacitySku": CAPACITY_SKU, "operationId": str(uuid.uuid4()), "operationName": op_name,
               "utilizationType": util_type, "status": "Success", "capacityUnitMs": cu_ms, "durationMs": dur_ms,
               "throttlingDelayMs": throttle_ms, "operationStartTime": iso(when - timedelta(milliseconds=dur_ms)),
               "identityType": "UPN", "identityValue": who, "releaseType": "Public"},
              f"/capacities/{CAP_ID}/workspaces/{d['workspaceId']}/items/{d['itemId']}", when)

def activity(user, act, ws, name, kind, when, method="Power BI web", ok=True):
    d = item_data(ws, name, kind)
    return ev("Estate.Activity", {"activity": act, "operation": act, "userId": user, "userType": "0",
              "workload": "PowerBI", **d, "consumptionMethod": method, "isSuccess": ok,
              "capacityId": CAP_ID, "capacityName": CAPACITY_NAME}, f"/workspaces/{d['workspaceId']}/items/{d['itemId']}", when)

# ---------------- simulation loop ----------------
def m(key):
    v = BEATS.get(key)
    return None if v is None else T0 + timedelta(minutes=v)

T0 = utcnow()
END = T0 + timedelta(minutes=DURATION_MINUTES)
spike_s, spike_e = m("capacity_spike_start"), m("capacity_spike_end")
burst_s, burst_e = m("export_burst_start"), m("export_burst_end")
fired, scheduled, overloaded = set(), [], False
next_window = T0.replace(second=(T0.second // 30) * 30, microsecond=0)
next_job = {"Load_Source_Data": T0 + timedelta(seconds=20), "Nightly combined load": T0 + timedelta(seconds=90),
            "refresh": T0 + timedelta(seconds=45)}

def utilization(t):
    base = 38 + 8 * math.sin((t - T0).total_seconds() / 240) + random.uniform(-3, 3)
    if spike_s and spike_e and spike_s <= t <= spike_e:
        span = (spike_e - spike_s).total_seconds()
        x = (t - spike_s).total_seconds() / span
        base += 85 * math.sin(math.pi * min(1, max(0, x)))        # rise and fall, peaking near 120%
    return max(5, base)

if SEND_BASELINE:
    print("Baseline:", sink.send(baseline()), "events")

total = 0
while utcnow() < END:
    now = utcnow()
    out = []
    # capacity summary every 30 s (+ state changes)
    while next_window + timedelta(seconds=30) <= now:
        u = utilization(next_window)
        delay = u * (1.05 if u > 90 else 0.8)
        cu = u / 100 * BASE_CU * 1000 * 30
        out.append(ev("Microsoft.Fabric.Capacity.Summary", {
            "capacityId": CAP_ID, "capacityName": CAPACITY_NAME, "capacitySku": CAPACITY_SKU,
            "windowStartTime": iso(next_window), "windowEndTime": iso(next_window + timedelta(seconds=30)),
            "baseCapacityUnits": BASE_CU, "capacityUnitMs": round(cu, 1),
            "utilizationInteractive": round(cu * (0.7 if u > 90 else 0.4), 1), "utilizationBackground": round(cu * (0.3 if u > 90 else 0.6), 1),
            "interactiveDelayThresholdPercentage": round(delay, 2), "interactiveRejectionThresholdPercentage": round(u * 0.55, 2),
            "backgroundRejectionThresholdPercentage": round(u * 0.25, 2), "overageTotalCapacityUnitMs": round(max(0, cu - BASE_CU * 30000) * 4, 1),
            "capacityRegion": "simulated"}, f"/capacities/{CAP_ID}", next_window + timedelta(seconds=30)))
        if delay > 100 and not overloaded:
            overloaded = True
            out.append(ev("Microsoft.Fabric.Capacity.State", {"capacityId": CAP_ID, "capacityName": CAPACITY_NAME, "capacitySku": CAPACITY_SKU,
                          "capacityState": "Overloaded", "stateChangeReason": "InteractiveDelay", "transitionTime": iso(next_window)}, f"/capacities/{CAP_ID}"))
        elif delay <= 95 and overloaded:
            overloaded = False
            out.append(ev("Microsoft.Fabric.Capacity.State", {"capacityId": CAP_ID, "capacityName": CAPACITY_NAME, "capacitySku": CAPACITY_SKU,
                          "capacityState": "Active", "stateChangeReason": "NotOverloaded", "transitionTime": iso(next_window)}, f"/capacities/{CAP_ID}"))
        next_window += timedelta(seconds=30)

    # steady background: viewers, Copilot, queries
    for _ in range(random.randint(1, 4)):
        who = upn(random.choice(VIEWERS))
        out.append(activity(who, "ViewReport", "3-PROD", REPORT, "Report", now))
        out.append(operation("3-PROD", REPORT, "SemanticModel", "Query", "Interactive",
                             random.uniform(800, 4000), random.randint(200, 1500), who, now))
    if random.random() < 0.25:
        out.append(activity(upn(random.choice(DIRECTORS)), "CopilotInteraction", "3-PROD", REPORT, "Report", now))
    if spike_s and spike_e and spike_s <= now <= spike_e:          # the slow visual's query storm
        for _ in range(6):
            out.append(operation("1-DEV", REPORT, "SemanticModel", "Query", "Interactive", random.uniform(60000, 140000),
                                 random.randint(4000, 12000), upn("mgmt_analyst"), now,
                                 throttle_ms=random.randint(500, 4000) if overloaded else 0))

    # recurring jobs
    if now >= next_job["Load_Source_Data"]:
        scheduled += job_events("1-DEV", "Load_Source_Data", "Notebook", "RunNotebook", "Scheduled", now, random.randint(70, 140))
        out.append(operation("1-DEV", "Load_Source_Data", "Notebook", "Notebook Scheduled Run", "Background", random.uniform(2e5, 5e5), 90000, upn("fabric_admin"), now))
        next_job["Load_Source_Data"] = now + timedelta(minutes=4)
    if now >= next_job["Nightly combined load"]:
        scheduled += job_events("3-PROD", "Nightly combined load", "DataPipeline", "Pipeline", "Scheduled", now, random.randint(40, 90))
        out.append(operation("3-PROD", "Nightly combined load", "DataPipeline", "Pipeline Run", "Background", random.uniform(1e5, 3e5), 60000, upn("fabric_admin"), now))
        next_job["Nightly combined load"] = now + timedelta(minutes=6)
    if now >= next_job["refresh"]:
        scheduled += job_events("3-PROD", REPORT, "SemanticModel", "Refresh", "Scheduled", now, random.randint(30, 60))
        out.append(operation("3-PROD", REPORT, "SemanticModel", "Dataset Scheduled Refresh", "Background", random.uniform(3e5, 6e5), 45000, upn("fabric_admin"), now))
        next_job["refresh"] = now + timedelta(minutes=5)

    # story beats
    t = m("refresh_failure")
    if t and now >= t and "refresh_failure" not in fired:
        fired.add("refresh_failure")
        scheduled += job_events("2-TEST-QA", REPORT, "SemanticModel", "Refresh", "OnDemand", now, 25, status="Failed",
                                error="Data source error: The database 'Combined-Data-Missing' was not found in the SQL endpoint.")
        print("Beat: refresh failure")
    if burst_s and burst_e and burst_s <= now <= burst_e and random.random() < 0.85:
        out.append(activity(upn("mgmt_analyst"), random.choice(["ExportReport", "ExportArtifact", "ExportReport"]), "3-PROD", REPORT, "Report", now))
        if "export_burst" not in fired:
            fired.add("export_burst"); print("Beat: export burst")
    t = m("new_admin")
    if t and now >= t and "new_admin" not in fired:
        fired.add("new_admin")
        out.append(ev("Estate.Access.Granted", {"workspaceId": ws_id("3-PROD"), "workspaceName": "3-PROD",
                      "principal": upn("contractor_temp"), "principalType": "User", "role": "Admin"}, f"/workspaces/{ws_id('3-PROD')}"))
        out.append(activity(upn("fabric_admin"), "UpdateWorkspaceAccess", "3-PROD", "3-PROD", "Workspace", now))
        print("Beat: new admin")
    t = m("content_changes")
    if t and now >= t and "content_changes" not in fired:
        fired.add("content_changes")
        d = item_data("1-DEV", "Executive Scorecard (copy)", "Report")
        out.append(ev("Estate.Inventory.ItemCreated", {**d, "workspaceType": "Workspace", "capacityId": CAP_ID, "state": "Active",
                      "lastUpdatedDate": iso(now), "createdBy": upn("mgmt_analyst")}, f"/workspaces/{d['workspaceId']}/items/{d['itemId']}"))
        d2 = item_data("3-PROD", REPORT, "SemanticModel")
        out.append(ev("Estate.Inventory.ItemUpdated", {**d2, "workspaceType": "Workspace", "capacityId": CAP_ID, "state": "Active",
                      "lastUpdatedDate": iso(now), "changedFields": "lastUpdatedDate"}, f"/workspaces/{d2['workspaceId']}/items/{d2['itemId']}"))
        out.append(activity(upn("mgmt_analyst"), "DeployArtifactsToPipelineStage", "3-PROD", REPORT, "SemanticModel", now))
        d3 = item_data("1-DEV", "Load_Source_Data", "Notebook")
        out.append(ev("Microsoft.Fabric.ItemUpdateSucceeded", {**d3, "executingPrincipalId": upn("mgmt_analyst")},
                      f"/workspaces/{d3['workspaceId']}/items/{d3['itemId']}"))
        print("Beat: content changes")

    # release scheduled job events that are due
    due = [e for (w, e) in scheduled if w <= now]
    scheduled = [(w, e) for (w, e) in scheduled if w > now]
    out += due
    total += sink.send(out)
    time.sleep(TICK_SECONDS)

if scheduled:
    total += sink.send([e for _, e in scheduled])
print(f"Simulation finished: {total} events over {DURATION_MINUTES} minutes.")
